# Day 062 — Solution: Testing Web Apps

In [ ]:
_TEST_APP_SRC = '"""test_app.py — Day 062: pytest test suite for a simple CRUD API.\n\nRun:  pytest test_app.py -v\n      pytest test_app.py -v -k "delete"   # filter by name\n      pytest test_app.py --tb=short        # compact tracebacks\n"""\nimport pytest\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nfrom starlette.testclient import TestClient\n\n\n# ── app under test ─────────────────────────────────────────────────────────────\nclass Item(BaseModel):\n    name:  str   = Field(min_length=1)\n    price: float = Field(gt=0)\n\n\ndef build_app() -> FastAPI:\n    """Simple item CRUD API — the subject under test."""\n    app  = FastAPI()\n    _db  = {}\n    _nxt = {"id": 1}\n\n    @app.get("/health")\n    def health():\n        return {"status": "ok"}\n\n    @app.get("/items")\n    def list_items():\n        return {"items": list(_db.values())}\n\n    @app.post("/items", status_code=201)\n    def create_item(item: Item):\n        iid         = _nxt["id"]\n        _nxt["id"] += 1\n        _db[iid]    = {"id": iid, **item.model_dump()}\n        return _db[iid]\n\n    @app.get("/items/{item_id}")\n    def get_item(item_id: int):\n        if item_id not in _db:\n            raise HTTPException(status_code=404, detail="Not found")\n        return _db[item_id]\n\n    @app.delete("/items/{item_id}", status_code=204)\n    def delete_item(item_id: int):\n        if item_id not in _db:\n            raise HTTPException(status_code=404, detail="Not found")\n        del _db[item_id]\n\n    return app\n\n\n# ── pytest fixture ─────────────────────────────────────────────────────────────\n@pytest.fixture\ndef client():\n    """Fresh TestClient (and fresh app state) for every test."""\n    return TestClient(build_app())\n\n\n# ── health tests ───────────────────────────────────────────────────────────────\ndef test_health_returns_200(client):\n    r = client.get("/health")\n    assert r.status_code == 200\n\n\ndef test_health_status_is_ok(client):\n    r = client.get("/health")\n    assert r.json()["status"] == "ok"\n\n\n# ── list tests ─────────────────────────────────────────────────────────────────\ndef test_list_items_empty_on_start(client):\n    r = client.get("/items")\n    assert r.status_code == 200\n    assert r.json()["items"] == []\n\n\n# ── create tests ───────────────────────────────────────────────────────────────\ndef test_create_item_returns_201(client):\n    r = client.post("/items", json={"name": "Widget", "price": 9.99})\n    assert r.status_code == 201\n\n\ndef test_create_item_has_id(client):\n    r = client.post("/items", json={"name": "Widget", "price": 9.99})\n    assert "id" in r.json()\n\n\ndef test_create_item_preserves_fields(client):\n    r = client.post("/items", json={"name": "Widget", "price": 9.99})\n    data = r.json()\n    assert data["name"]  == "Widget"\n    assert data["price"] == 9.99\n\n\n@pytest.mark.parametrize("name,price,expected_status", [\n    ("Widget", 9.99,  201),   # valid\n    ("",       9.99,  422),   # empty name\n    ("Widget", 0.0,   422),   # price must be > 0\n    ("Widget", -1.0,  422),   # negative price\n])\ndef test_create_item_validation(client, name, price, expected_status):\n    r = client.post("/items", json={"name": name, "price": price})\n    assert r.status_code == expected_status, (\n        f"name={name!r}, price={price}: expected {expected_status}, got {r.status_code}")\n\n\n# ── get tests ──────────────────────────────────────────────────────────────────\ndef test_get_item_after_create(client):\n    created = client.post("/items", json={"name": "Gadget", "price": 14.99}).json()\n    r = client.get(f"/items/{created[\'id\']}")\n    assert r.status_code == 200\n    assert r.json()["name"] == "Gadget"\n\n\ndef test_get_item_not_found(client):\n    r = client.get("/items/999")\n    assert r.status_code == 404\n\n\n# ── delete tests ───────────────────────────────────────────────────────────────\ndef test_delete_item_returns_204(client):\n    created = client.post("/items", json={"name": "Thing", "price": 1.0}).json()\n    r = client.delete(f"/items/{created[\'id\']}")\n    assert r.status_code == 204\n\n\ndef test_deleted_item_is_gone(client):\n    created = client.post("/items", json={"name": "Thing", "price": 1.0}).json()\n    client.delete(f"/items/{created[\'id\']}")\n    r = client.get(f"/items/{created[\'id\']}")\n    assert r.status_code == 404\n\n\ndef test_delete_item_not_found(client):\n    r = client.delete("/items/999")\n    assert r.status_code == 404\n'
from pathlib import Path
Path('test_app.py').write_text(_TEST_APP_SRC)
print('test_app.py written.')

In [ ]:
# inline verification — demonstrates all patterns without running pytest subprocess
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

class Item(BaseModel):
    name:  str   = Field(min_length=1)
    price: float = Field(gt=0)

def build_app():
    app = FastAPI(); _db = {}; _nxt = {"id": 1}
    @app.get("/health")
    def health(): return {"status": "ok"}
    @app.get("/items")
    def list_items(): return {"items": list(_db.values())}
    @app.post("/items", status_code=201)
    def create(item: Item):
        iid = _nxt["id"]; _nxt["id"] += 1
        _db[iid] = {"id": iid, **item.model_dump()}; return _db[iid]
    @app.get("/items/{item_id}")
    def get(item_id: int):
        if item_id not in _db: raise HTTPException(404)
        return _db[item_id]
    @app.delete("/items/{item_id}", status_code=204)
    def delete(item_id: int):
        if item_id not in _db: raise HTTPException(404)
        del _db[item_id]
    return app

def fresh(): return TestClient(build_app(), raise_server_exceptions=False)

# health
c = fresh()
assert c.get("/health").json()["status"] == "ok"
print("\u2705 /health")

# list empty
c = fresh()
assert c.get("/items").json()["items"] == []
print("\u2705 list empty")

# create
c = fresh()
r = c.post("/items", json={"name": "Widget", "price": 9.99})
assert r.status_code == 201 and r.json()["name"] == "Widget" and "id" in r.json()
print("\u2705 create item 201")

# validation
c = fresh()
assert c.post("/items", json={"name": "", "price": 9.99}).status_code == 422
assert c.post("/items", json={"name": "X", "price": 0}).status_code == 422
assert c.post("/items", json={"name": "X", "price": -1}).status_code == 422
print("\u2705 validation 422 cases")

# get
c = fresh()
created = c.post("/items", json={"name": "G", "price": 1.0}).json()
assert c.get(f"/items/{created['id']}").json()["name"] == "G"
assert c.get("/items/999").status_code == 404
print("\u2705 get item / 404")

# delete
c = fresh()
created = c.post("/items", json={"name": "T", "price": 1.0}).json()
assert c.delete(f"/items/{created['id']}").status_code == 204
assert c.get(f"/items/{created['id']}").status_code == 404
assert c.delete("/items/999").status_code == 404
print("\u2705 delete item / 404")

# parametrize-style: multiple validation cases
cases = [("Widget", 9.99, 201), ("", 9.99, 422), ("X", 0.0, 422), ("X", -1, 422)]
c = fresh()
for name, price, expected in cases:
    actual = c.post("/items", json={"name": name, "price": price}).status_code
    assert actual == expected, f"name={name!r}, price={price}: expected {expected}, got {actual}"
print("\u2705 parametrized validation cases")

print("\nDay 062 \u2014 Testing Web Apps complete! \U0001f389")
